In [1]:
from src.dataset_manager import DatasetManager
from src.models.negative_binomial import NegativeBinomialPiecewise
from src.training_manager import GGSTrainingManager
from src.test_manager import TestManager

/home/jdani/proyects/Premant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
m_train, m_test = DatasetManager.split_dataset()

Training engines: 140
Test engines: 60


In [3]:
Training = GGSTrainingManager(
    model=NegativeBinomialPiecewise(),
    list_ids=m_train
)

Test = TestManager(
    model=NegativeBinomialPiecewise(),
    train_ids=m_train,
    test_ids=m_test
)

# First GSS - Exploratory Search

In [4]:
param_grid = {
    'alpha': [0.1, 0.5, 1.0, 1.5],
    'link_type': ['log'], 
    'alpha_reg': [0.0, 0.05],            
    'l1_ratio': [0.0, 0.5],              
    'clipping_threshold': [115, 120, 125, 130]              
}

ggs = Training.group_grid_search(param_grid=param_grid)
display(Training.get_ggs_results())

Starting Grid Search: 64 configs × 5 folds = 320 tasks


GGS progress: 100%|██████████| 64/64 [25:51<00:00, 24.25s/it]  


,alpha,link_type,alpha_reg,l1_ratio,clipping_threshold,mean_S_score,mean_C_index,mean_MAE,mean_RMSE,Success
0,0.5,log,0.00,0.0,115,5.954831,0.859447,12.775645,18.256651,1
1,0.5,log,0.00,0.5,115,5.954831,0.859447,12.775645,18.256651,1
2,1.0,log,0.00,0.0,115,5.959046,0.859379,12.767049,18.267563,1
3,1.0,log,0.00,0.5,115,5.959046,0.859379,12.767049,18.267563,1
4,1.5,log,0.00,0.0,115,5.961229,0.859323,12.763686,18.271364,1
5,1.5,log,0.00,0.5,115,5.961229,0.859323,12.763686,18.271364,1
6,0.1,log,0.00,0.0,115,5.969582,0.860240,12.824017,18.200994,1
7,0.1,log,0.00,0.5,115,5.969582,0.860240,12.824017,18.200994,1
8,0.1,log,0.05,0.0,115,5.987947,0.861009,13.536601,18.901325,1
9,0.1,log,0.05,0.5,115,6.448323,0.857754,13.485691,18.883902,1


In [5]:
display(Training.get_ggs_results(100))

,alpha,link_type,alpha_reg,l1_ratio,clipping_threshold,mean_S_score,mean_C_index,mean_MAE,mean_RMSE,Success
0,0.5,log,0.00,0.0,115,5.954831,0.859447,12.775645,18.256651,1
1,0.5,log,0.00,0.5,115,5.954831,0.859447,12.775645,18.256651,1
2,1.0,log,0.00,0.0,115,5.959046,0.859379,12.767049,18.267563,1
3,1.0,log,0.00,0.5,115,5.959046,0.859379,12.767049,18.267563,1
4,1.5,log,0.00,0.0,115,5.961229,0.859323,12.763686,18.271364,1
...,...,...,...,...,...,...,...,...,...,...
59,1.5,log,0.05,0.5,130,17.765802,0.836743,23.703765,29.816461,1
60,1.0,log,0.05,0.0,130,19.286210,0.838101,23.016919,29.777963,1
61,1.5,log,0.05,0.0,120,19.689985,0.851875,25.481120,31.543461,1
62,1.5,log,0.05,0.0,125,23.927896,0.844116,26.378929,32.834420,1


# Second GSS - Detailed Search

In [ ]:
# Búsqueda de hiperparámetros

param_grid_detailed = {
    'model__link_type': ['log'],           
    'model__alpha': [1.2, 1.5, 1.8],             
    'model__alpha_reg': [0.01, 0.03, 0.05, 0.07], 
    'model__l1_ratio': [0.85, 0.95, 1.0],        
    'model__clipping_threshold': [110, 115, 120] 
}          

ggs_detail = Training.group_grid_search(param_grid=param_grid_detailed)
display(Training.get_ggs_results())

# Final Test - Evaluation With Tune Parameters

In [3]:
# Evaluación final

param_grid_tune = {
    'model__link_type': 'log',           
    'model__alpha': 1.5,             
    'model__alpha_reg': 0.05, 
    'model__l1_ratio': 1.0,       
    'model__clipping_threshold': 110 
}     

results = Test.evaluate_best_model(param_grid=param_grid_tune)
display(results)

,Partition,Mode,N,S-Score,C-Index,MAE,RMSE
0,Train,Trajectory,23824,6.485935,0.859662,17.116341,21.159620
1,Train,Deployment,140,14.612286,0.920408,21.841584,24.811599
2,Test,Trajectory,9903,8.294493,0.845016,17.876341,22.465329
3,Test,Deployment,60,20.126744,0.922168,23.170191,26.616462
